In [ ]:
#!/usr/bin/env python
"""
Session Segmentation Pipeline

Converts the raw per-turn tutoring session log into dialogue segments used
throughout the analysis. A segment is the speech occurring between two
consecutive logged events, paired with the event that most recently
preceded it (the "event context").

Input:  all_sessions_segments.csv
        Raw per-turn log with columns: transcriptID, speaker, speaker_type,
        timestamp, type ('speech' or 'event'), description

Output: session_level_dialog_segments_final.csv
        One row per dialogue segment, with columns:
        transcriptID, segment_id, event_context, dialog (preceding context,
        concatenated from the prior 3 segments), next_tutor_utterance,
        next_tutor_utterance_n1, next_tutor_utterance_n2

Run once, before the LLM generation step (03_generate_llm_responses.py) and
the topic extraction step (02_extract_topics.py), both of which consume
this script's output file.
"""

import pandas as pd

# ── Config -------------------------------------------------------------------
# Change these if your file names or segmentation parameters differ.

RAW_LOG_PATH = "all_sessions_segments.csv"
OUTPUT_PATH = "session_level_dialog_segments_final.csv"

# How many of a segment's tutor speech lines get pulled into
# "next_tutor_utterance" (the ground-truth target the LLM is trying to match).
MAX_TUTOR_LINES_PER_SEGMENT = 4

# How many PRIOR segments get concatenated to form the "dialog" field
# (the preceding context shown to the LLM before it generates).
N_PRIOR_SEGMENTS_FOR_CONTEXT = 3


def anonymize_role(speaker_type: str) -> str:
    """
    Collapse whatever the raw speaker label says (e.g. "Tutor Ghaffar",
    a student's first name) down to just "Tutor" or "Student", so no
    real names end up in the dialogue text we later send to an LLM.
    """
    return "Tutor" if "tutor" in speaker_type.lower() else "Student"


def build_segments(df: pd.DataFrame) -> pd.DataFrame:
    """
    Walk each session in chronological order and group speech into segments.

    The core idea: every time we hit an 'event' row (e.g. "draws with a red
    pen"), that event becomes a segmentation boundary. All the speech rows
    we've accumulated SINCE the last boundary get bundled into one segment,
    tagged with whichever event most recently preceded them.

    Example, for one session:
        row 1: event   "joined the session"
        row 2: speech  "Hi!"                    }
        row 3: speech  "Let's start."            } -> segment 0, event_context = "joined the session"
        row 4: event   "draws with a red pen"
        row 5: speech  "Look at this."           } -> segment 1, event_context = "draws with a red pen"
        (end of session, segment 1 gets flushed)
    """
    # Remove exact duplicate rows (can happen from how the raw log was
    # exported) and put everything in strict chronological order per session.
    df = df.drop_duplicates(subset=["transcriptID", "timestamp", "type", "description"])
    df = df.sort_values(["transcriptID", "timestamp"]).reset_index(drop=True)

    segments = []          # the finished segments we've built so far
    current_transcript = None   # which session we're currently walking through
    current_segment_id = 0      # segment counter WITHIN the current session
    current_text = []           # speech lines accumulated since the last event
    last_event = None           # the most recent event description seen

    for _, row in df.iterrows():

        # ── Case 1: we've moved on to a new session ─────────────────────────
        # Flush whatever partial segment we were building for the PREVIOUS
        # session, then reset all the per-session tracking variables.
        if row["transcriptID"] != current_transcript:
            if current_text:
                segments.append({
                    "transcriptID": current_transcript,
                    "segment_id": current_segment_id,
                    "event_context": last_event,
                    "dialog": " [SEP] ".join(current_text),
                })
            current_transcript = row["transcriptID"]
            current_segment_id = 0
            current_text = []
            last_event = None

        # ── Case 2: this row is an event -> it closes the current segment ──
        if row["type"] == "event":
            # Only flush a segment if there's actually speech to bundle up.
            # (Two events in a row with no speech between them just means
            # the "last_event" gets overwritten below, no new segment yet.)
            if current_text:
                segments.append({
                    "transcriptID": current_transcript,
                    "segment_id": current_segment_id,
                    "event_context": last_event,
                    "dialog": " [SEP] ".join(current_text),
                })
                current_segment_id += 1
                current_text = []
            last_event = row["description"]

        # ── Case 3: this row is speech -> accumulate it into the current segment ──
        elif row["type"] == "speech":
            role = anonymize_role(row["speaker_type"])
            current_text.append(f"{role}: {row['description']}")

    # After the loop ends, there's likely one final segment still "open"
    # (the speech after the very last event of the very last session) --
    # flush that one too, or it would silently get dropped.
    if current_text:
        segments.append({
            "transcriptID": current_transcript,
            "segment_id": current_segment_id,
            "event_context": last_event,
            "dialog": " [SEP] ".join(current_text),
        })

    return pd.DataFrame(segments)


def extract_next_tutor_utterance(dialog: str, max_lines: int = MAX_TUTOR_LINES_PER_SEGMENT):
    """
    Given a segment's raw dialog string (multiple "Tutor: ..." / "Student: ..."
    lines joined by " [SEP] "), pull out just the tutor's lines, up to
    `max_lines` of them. This becomes the ground-truth target we compare
    LLM generations against later.

    Returns None if there's no tutor speech in this segment at all -- those
    segments get dropped later, since there's nothing to score an LLM
    generation against.
    """
    if pd.isna(dialog):
        return None

    tutor_lines = []
    for part in dialog.split("[SEP]"):
        part = part.strip()
        if part.lower().startswith("tutor"):
            # Strip the "Tutor: " prefix, keep just the actual words.
            text = part.split(":", 1)[1].strip() if ":" in part else part
            tutor_lines.append(text)
            if len(tutor_lines) == max_lines:
                break  # stop once we've collected enough lines

    return " ".join(tutor_lines) if tutor_lines else None


def add_context_and_targets(segments_df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the raw segments (one row per event boundary) and turns each one
    into a full training/evaluation example:

      - next_tutor_utterance / _n1 / _n2 : what the tutor ACTUALLY said next
        (and the two utterances after that, for supplementary analyses of
        whether an LLM generation anticipates later dialogue).

      - dialog : replaced with the CONCATENATED PRIOR N segments, i.e. the
        conversational history an LLM would need to see to generate a
        plausible next line. (Before this function runs, "dialog" just
        holds that ONE segment's own speech -- we overwrite it here.)

    Also drops segments that can't actually be used:
      - no tutor utterance to compare against (dialog had no "Tutor:" lines)
      - no prior context available yet (this is the very first segment of
        a session, so there's nothing to build "dialog" out of)
    """
    df = segments_df.sort_values(["transcriptID", "segment_id"]).reset_index(drop=True)

    # ── Step 1: pull out the ground-truth targets from each segment's own speech ──
    df["next_tutor_utterance"] = df["dialog"].apply(extract_next_tutor_utterance)
    # .shift(-1) / .shift(-2), grouped by session, give us "the same field but
    # from 1 (or 2) segments later in this same session" -- used only for
    # supplementary "does the model anticipate future dialogue" analyses.
    df["next_tutor_utterance_n1"] = df.groupby("transcriptID")["next_tutor_utterance"].shift(-1)
    df["next_tutor_utterance_n2"] = df.groupby("transcriptID")["next_tutor_utterance"].shift(-2)

    # ── Step 2: build the "prior N segments" columns we'll use as context ──
    # .shift(k) grouped by session gives us "this same field, but from k
    # segments EARLIER in this session" -- i.e., exactly what came before.
    prior_cols = []
    for k in range(1, N_PRIOR_SEGMENTS_FOR_CONTEXT + 1):
        col = f"prev_{k}"
        df[col] = df.groupby("transcriptID")["dialog"].shift(k)
        prior_cols.append(col)

    def combine_prior_context(row):
        # Concatenate whichever prior segments actually exist, in
        # chronological order (oldest first), skipping any that are
        # missing (e.g. segment 2 only has 1 prior segment, not 3).
        parts = [row[c] for c in reversed(prior_cols) if pd.notna(row[c])]
        return " [CTX] ".join(parts) if parts else None

    # Overwrite "dialog" -- it now means "the preceding context", not
    # "this segment's own speech" (which we already extracted above).
    df["dialog"] = df.apply(combine_prior_context, axis=1)
    df = df.drop(columns=prior_cols)  # no longer needed, already combined

    # ── Step 3: drop segments we can't actually use ──────────────────────
    n_start = len(df)

    # Reason 1: no tutor line to compare an LLM generation against.
    df = df.dropna(subset=["next_tutor_utterance"])
    n_after_utterance_filter = len(df)

    # Reason 2: no prior segments exist yet (this IS the first segment of
    # its session, so combine_prior_context() above returned None for it).
    df = df.dropna(subset=["dialog"])
    n_final = len(df)

    print(f"Segments before filtering: {n_start}")
    print(f"Excluded for missing next tutor utterance: {n_start - n_after_utterance_filter} "
          f"({100 * (n_start - n_after_utterance_filter) / n_start:.1f}%)")
    print(f"Excluded for missing prior context (first segment of a session): "
          f"{n_after_utterance_filter - n_final} "
          f"({100 * (n_after_utterance_filter - n_final) / n_start:.1f}%)")
    print(f"Final segment count: {n_final}")

    return df[["transcriptID", "segment_id", "event_context", "dialog",
               "next_tutor_utterance", "next_tutor_utterance_n1", "next_tutor_utterance_n2"]]


def main():
    raw_df = pd.read_csv(RAW_LOG_PATH)

    print("Step 1/2: Building segments from raw log...")
    segments_df = build_segments(raw_df)
    print(f"  {segments_df['transcriptID'].nunique()} sessions, {len(segments_df)} candidate segments")

    print("\nStep 2/2: Adding context and applying exclusion filters...")
    final_df = add_context_and_targets(segments_df)

    final_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()